# Data Consistency

Data consistency means ensuring that data follows **expected rules, valid ranges, standard formats, and logical relationships**.

Main checks:

1. Invalid Values
2. Unit Consistency
3. Duplicate Identifiers

---

# 1. INVALID VALUES

An **invalid value** does not follow the expected rules or valid range of a column.

Example:

```text
Age:

25
30
-5
150
````

`-5` and `150` may be invalid depending on the dataset and domain rules.

### Check Invalid Values

Values below 0:

```python
df[df["Age"] < 0]
```

Values above 100:

```python
df[df["Age"] > 100]
```

Check both conditions:

```python
df[
    (df["Age"] < 0) |
    (df["Age"] > 100)
]
```

---

## Categorical Value Checks

Check all unique values:

```python
df["Gender"].unique()
```

Check the frequency of each value:

```python
df["Gender"].value_counts()
```

Example:

```text
Male       → 50
Female     → 45
Unknown    → 3
M           → 2
```

This can reveal inconsistent or unexpected categories such as:

```text
Male
male
M
MALE
```

These may represent the same category but are stored differently.

---

## Bounded Numerical Values

Some numerical variables have a valid range.

Example: Rating should be between 1 and 5.

```python
df[
    (df["Rating"] < 1) |
    (df["Rating"] > 5)
]
```

This returns ratings outside the expected range.

### Important

> **Invalid values depend on domain rules.**

A value that is invalid in one dataset may be completely valid in another.

Always understand the meaning and expected range of the variable before declaring a value invalid.

---

# 2. UNIT CONSISTENCY

**Unit consistency** means ensuring that the same type of measurement uses the **same unit** throughout the dataset.

Values must use the same unit before performing calculations or comparisons.

Example:

```text
Height:

170 cm
1.75 m
180 cm
```

These values use different units.

Convert them to one standard unit.

```text
1.75 m = 175 cm
```

After conversion:

```text
170 cm
175 cm
180 cm
```

Now the values can be compared correctly.

---

## Weight Example

Suppose the dataset contains:

```text
70 kg
75 kg
154 lb
```

Convert everything to one unit.

Approximately:

```text
154 lb ≈ 69.85 kg
```

Then:

```text
70 kg
75 kg
69.85 kg
```

---

## General Process

```text
Check Units
    ↓
Choose Standard Unit
    ↓
Convert All Values
    ↓
Validate Converted Values
    ↓
Perform Calculations
```

### Important

Never directly calculate statistics or compare values when the same column contains **mixed units**.

For example, calculating the mean of:

```text
170 cm
1.75 m
180 cm
```

without conversion would produce an incorrect result.

---

# 3. DUPLICATE IDENTIFIERS

Identifiers are columns used to uniquely identify records.

Examples:

* Customer_ID
* Employee_ID
* Order_ID
* Transaction_ID
* Product_ID

Some identifiers are expected to be unique.

---

## Check Whether an Identifier Is Unique

Use:

```python
df["Customer_ID"].is_unique
```

Result:

```text
True
```

→ All values are unique.

```text
False
```

→ Duplicate values exist.

---

## Find Duplicated Identifiers

Use:

```python
df["Customer_ID"].duplicated()
```

This returns `True` for duplicated values.

---

## Find Duplicate Records

```python
df[
    df["Customer_ID"].duplicated()
]
```

This returns rows where the identifier is duplicated after its first occurrence.

---

## Find All Rows Containing Duplicated IDs

Use:

```python
df[
    df["Customer_ID"].duplicated(keep=False)
]
```

`keep=False` marks **all occurrences** of duplicated identifiers as `True`.

Example:

```text
Customer_ID

101
102
101
103
102
```

Result:

```text
101 → duplicated
102 → duplicated
101 → duplicated
102 → duplicated
```

---

## Count Unique Identifiers

```python
df["Customer_ID"].nunique()
```

This returns the number of unique identifier values.

---

## Important: Duplicate ID ≠ Automatically an Error

A duplicate identifier is **not automatically an error**.

It depends on what each row represents.

Example:

If each row represents a **customer**:

```text
Customer_ID
101
102
103
```

The ID may be expected to be unique.

But if each row represents a **transaction**:

```text
Customer_ID
101
101
101
102
102
```

The same customer can legitimately appear multiple times because one customer can make multiple transactions.

Therefore:

> Always understand the **row-level meaning** before removing duplicate identifiers.

---

# DATA CONSISTENCY CHECKLIST

Before using a dataset:

1. Check valid numerical ranges
2. Check allowed categorical values
3. Check for inconsistent categories
4. Check measurement units
5. Choose and standardize units
6. Check identifier uniqueness when required
7. Investigate inconsistencies
8. Correct or remove values only when justified

---

# 🧠 KEY IDEA

> **Data Consistency = Values follow the expected rules and represent information in a consistent way.**

```text
Invalid Values
      ↓
Check Rules & Ranges

Unit Consistency
      ↓
Standardize Units

Duplicate Identifiers
      ↓
Check Based on Row Meaning
```

Always use **domain knowledge** before deciding that a value or duplicate is actually incorrect.

```
```


In [1]:
import pandas as pd
data = {
    "Customer_ID": [
        101, 102, 103, 103, 104, 105
    ],

    "Age": [
        25, 32, -5, 45, 150, 28
    ],

    "Gender": [
        "Male",
        "Female",
        "M",
        "Female",
        "Unknown",
        "Male"
    ],

    "Height": [
        "170 cm",
        "1.75 m",
        "180 cm",
        "1.65 m",
        "175 cm",
        "1.80 m"
    ],

    "Weight": [
        "70 kg",
        "75 kg",
        "154 lb",
        "80 kg",
        "165 lb",
        "72 kg"
    ]
}

df = pd.DataFrame(data)

In [2]:
df

,Customer_ID,Age,Gender,Height,Weight
0,101,25,Male,170 cm,70 kg
1,102,32,Female,1.75 m,75 kg
2,103,-5,M,180 cm,154 lb
3,103,45,Female,1.65 m,80 kg
4,104,150,Unknown,175 cm,165 lb
5,105,28,Male,1.80 m,72 kg


In [8]:
# age cannot be -5 and and age cannot be 150
df['Age'] = df['Age'].astype('int')
df = df.drop(df[(df['Age']>100) | (df['Age']<0)].index)

In [9]:
df

,Customer_ID,Age,Gender,Height,Weight
0,101,25,Male,170 cm,70 kg
1,102,32,Female,1.75 m,75 kg
3,103,45,Female,1.65 m,80 kg
5,105,28,Male,1.80 m,72 kg


In [10]:
df['Height'] = df['Height'].apply( lambda x: float(x.replace('cm','')) if x.endswith('cm') else float(x.replace('m',''))*100)

In [11]:
df

,Customer_ID,Age,Gender,Height,Weight
0,101,25,Male,170.0,70 kg
1,102,32,Female,175.0,75 kg
3,103,45,Female,165.0,80 kg
5,105,28,Male,180.0,72 kg


In [12]:
df.dtypes

Customer_ID      int64
Age              int64
Gender             str
Height         float64
Weight             str
dtype: object

In [13]:
df['Weight'] = df['Weight'].apply(lambda x: float(x.replace('kg','')))

In [14]:
df

,Customer_ID,Age,Gender,Height,Weight
0,101,25,Male,170.0,70.0
1,102,32,Female,175.0,75.0
3,103,45,Female,165.0,80.0
5,105,28,Male,180.0,72.0


In [17]:
df.dtypes

Customer_ID      int64
Age              int64
Gender             str
Height         float64
Weight         float64
dtype: object